In [16]:
from langchain_community.vectorstores.cassandra import Cassandra
from langchain_classic.indexes.vectorstore import VectorStoreIndexWrapper
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

In [17]:
from datasets import load_dataset

# with cassio, the engine powering the astra DB integration in Lnagchain
#u will also initialize the DB connection:
import cassio
from PyPDF2 import PdfReader

In [18]:
from dotenv import load_dotenv
load_dotenv()

True

In [19]:
pdfreader=PdfReader(r"D:\AI\LangChain\Project\research_papers\Attention.pdf")

In [20]:
from typing_extensions import Concatenate
# read the text from the pdf
raw_text=""
for i,page in enumerate(pdfreader.pages):
    content=page.extract_text()
    if content:
        raw_text+=content

In [21]:
# Initialize the connection to your database
import os
cassio.init(token=os.getenv("ASTRA_DB_APPLICATION_TOKEN"),database_id=os.getenv("ASTRA_DB_ID"))

In [22]:
model=ChatGroq(model="llama-3.3-70b-versatile")
embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8572.20it/s]


In [23]:
# create the vectorstore
astra_vectorstore=Cassandra(
    embedding=embeddings,
    table_name="qa_mini",
    session=None,
    keyspace=None
) 

In [24]:
from langchain_classic.text_splitter import CharacterTextSplitter
text_splitter=CharacterTextSplitter(
    separator='\n',
    chunk_size=800,
    chunk_overlap=200,
    length_function=len,
)
chunks=text_splitter.split_text(raw_text)

In [25]:
astra_vectorstore.add_texts(chunks)
astra_vector_index=VectorStoreIndexWrapper(vectorstore=astra_vectorstore)

In [26]:
# Testing
first_question=True
while True:
    if first_question:
        query=input("\nEnter your query (or type 'quit' to exit): ").strip()
    else:
        query=input("\nWhat's your next question (or type 'quit' to exit): ").strip()

    if query.lower()=='quit':
        break
    if query=="":
        continue
    first_question=False
    print("\nQuestion: \"%s\""%query)
    answer=astra_vector_index.query(query,llm=model).strip()
    print("\nAnswer: \"%s\"\n"%answer )

    print("FIRST DOCUMENTS BY RELEVANCE:")
    for doc,score in astra_vectorstore.similarity_search_with_score(query,k=4):
        print("   [%0.4f]\"%s...\""%(score,doc.page_content))


Question: "What is Transformer"

Answer: "The Transformer is a neural network architecture introduced in the context of natural language processing, specifically for sequence-to-sequence tasks such as machine translation. It follows an encoder-decoder structure, using stacked self-attention and point-wise, fully connected layers for both the encoder and decoder.

The encoder takes in a sequence of symbols and generates a continuous representation, which is then used by the decoder to generate an output sequence one element at a time. The decoder is auto-regressive, meaning it consumes the previously generated symbols as additional input when generating the next symbol.

The Transformer architecture consists of an encoder stack with 6 identical layers, each having two sub-layers: a multi-head self-attention mechanism and a simple, position-wise fully connected layer. The decoder has a similar structure. 

The Transformer is designed to handle long-range dependencies in input sequences 